# 03 - XGBoost Training + SHAP

This notebook mirrors `backend/features.py`, `backend/xgboost_model.py`, and SHAP usage.

In [ ]:
import sys
from pathlib import Path
import duckdb
import pandas as pd
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import shap

sys.path.append(str(Path("..").resolve()))
from backend.features import make_features

In [ ]:
ticker = "AAPL"
conn = duckdb.connect("../data/finvizaard.duckdb")
prices = conn.execute("""
SELECT ts, open, high, low, close, volume
FROM prices
WHERE ticker = ?
ORDER BY ts
""", [ticker]).df()

feats = make_features(prices).dropna().copy()
feature_cols = [
    "ret_1", "ret_5", "vol_5", "vol_20",
    "ma_5", "ma_20", "ma_ratio", "range", "vol_chg"
]
X = feats[feature_cols]
y = feats["y"]
len(feats)

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = XGBRegressor(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
)
model.fit(X_train, y_train)
pred_test = model.predict(X_test)
mae = mean_absolute_error(y_test, pred_test)
print("Test MAE:", round(float(mae), 4))

In [ ]:
X_latest = X.tail(1)
explainer = shap.TreeExplainer(model, data=X_train.tail(min(200, len(X_train))), feature_names=feature_cols)
shap_vals = explainer.shap_values(X_latest)
contrib = {feature_cols[i]: float(shap_vals[0][i]) for i in range(len(feature_cols))}
contrib

## Demo Notes
- Mention train/test split and MAE for transparency.
- Show SHAP contributions to explain one latest prediction.